# 02 — Bias Analysis & Fairness Assessment

**Project:** NovaCred Credit Application Governance Audit  
**Role:** Data Scientist  
**Notebook:** Bias detection, disparate impact analysis, and fairness metrics  

---

## Objective

This notebook audits the NovaCred credit application dataset for algorithmic bias and discriminatory lending patterns. The analysis covers:

1. **Disparate Impact (DI) ratio** for gender — four-fifths rule (DI < 0.8 = potential violation)
2. **Age-based bias** in approval rates and interest rates
3. **Proxy discrimination** — non-protected features acting as stand-ins for protected attributes
4. **Interaction effects** between protected attributes
5. **Fairlearn metrics** — demographic parity difference

> **Data contract:** All data is loaded through `src/data_loading.load_raw_data`, which returns a flat DataFrame with raw values preserved. No cleaning is performed at load time — all pre-processing for fairness analysis is done explicitly in this notebook.

## 1. Setup

Configure logging to surface any warnings raised by the data loader (e.g. skipped records), set the random seed for reproducibility, and import all libraries used in this notebook.

In [1]:
import logging
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

# Make src/ importable when running from the notebooks/ directory
sys.path.insert(0, str(Path("..").resolve()))

from src.data_loading import load_raw_data

# -- Logging ------------------------------------------------------------------
# Configure root logger so that INFO/WARNING messages from src.data_loading
# are visible in the notebook output.
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s | %(name)s | %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger("02-bias-analysis")

# -- Reproducibility ----------------------------------------------------------
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# -- Plot style ---------------------------------------------------------------
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
FIGURES_DIR = Path("../reports/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

log.info("Setup complete. Figures will be saved to: %s", FIGURES_DIR.resolve())

INFO | 02-bias-analysis | Setup complete. Figures will be saved to: /Users/niklasduttmann/Desktop/NOVA/T3 - Data Ecosystems and Governance in Organizations/Assignment/Code/dego-2606-group3/reports/figures


## 2. Data Loading

`load_raw_data` reads the nested JSON file and returns a **flat DataFrame** — one row per application. Nested objects (`applicant_info`, `financials`, `decision`) are unpacked into prefixed columns. Spending categories are pivoted into `spending_<category>` columns.

No cleaning is applied by the loader. All raw inconsistencies (mixed types, duplicate records, inconsistent gender coding, mixed date formats) are preserved and will be handled explicitly below before any group-level statistics are computed.

In [2]:
DATA_PATH = "../data/raw/raw_credit_applications.json"

df_raw = load_raw_data(DATA_PATH)

log.info("Loaded dataset: %d rows x %d columns", *df_raw.shape)

INFO | src.data_loading | Loaded 502 raw records from '../data/raw/raw_credit_applications.json'.
INFO | src.data_loading | Successfully flattened 502/502 records.
INFO | 02-bias-analysis | Loaded dataset: 502 rows x 34 columns


## 3. Initial Inspection

Before any bias analysis, confirm the shape of the loaded DataFrame, verify that key columns are present, and get a first look at dtypes and missing values. This also acts as a sanity check that `load_raw_data` produced the expected output.

In [3]:
# Columns required for bias analysis — raise early if any are missing
REQUIRED_COLS = ["gender", "date_of_birth", "zip_code", "loan_approved", "interest_rate"]
missing_cols = [c for c in REQUIRED_COLS if c not in df_raw.columns]
if missing_cols:
    raise KeyError(f"Required columns missing from dataset: {missing_cols}")

print(f"Shape: {df_raw.shape}")
print(f"\nColumn dtypes:")
print(df_raw.dtypes.to_string())

Shape: (502, 34)

Column dtypes:
id                                  str
processing_timestamp                str
loan_purpose                        str
notes                               str
full_name                           str
email                               str
ssn                                 str
ip_address                          str
gender                              str
date_of_birth                       str
zip_code                            str
annual_income                    object
credit_history_months             int64
debt_to_income                  float64
savings_balance                   int64
spending_shopping               float64
spending_rent                   float64
spending_alcohol                float64
loan_approved                      bool
interest_rate                   float64
approved_amount                 float64
rejection_reason                    str
spending_dining                 float64
spending_healthcare             float64
spendin

In [4]:
# Missing value counts for columns relevant to fairness analysis
fairness_cols = [
    "gender", "date_of_birth", "zip_code",
    "annual_income", "loan_approved", "interest_rate", "rejection_reason",
]

missing_summary = (
    df_raw[fairness_cols]
    .isnull()
    .sum()
    .rename("missing_count")
    .to_frame()
    .assign(missing_pct=lambda x: (x["missing_count"] / len(df_raw) * 100).round(2))
)

print("Missing values in fairness-relevant columns:")
display(missing_summary)

Missing values in fairness-relevant columns:


,missing_count,missing_pct
gender,1,0.20
date_of_birth,1,0.20
zip_code,1,0.20
annual_income,5,1.00
loan_approved,0,0.00
interest_rate,210,41.83
rejection_reason,292,58.17


In [5]:
# Raw gender value counts — expected to reveal inconsistent coding
# (e.g. 'male', 'Male', 'M' all referring to the same group)
print("Raw 'gender' value counts (before normalisation):")
print(df_raw["gender"].value_counts(dropna=False).to_string())

Raw 'gender' value counts (before normalisation):
gender
Male      195
Female    193
F          58
M          53
            2
NaN         1
